# Project 5 - Three Custom Middleware

#### Objective: Write one middleware of each style and show what each changes about the agent's behaviour.

##### Test
```Ask a normal question in 'fr' mode; ask a question containing a banned word; ask a normal question in '```

##### Expected Output
```
[timing] model=gpt-... latency=812ms tokens=310
Bonjour ! ...
[guardrail] blocked - no model call made
Sorry, I can't help with that topic.
```

#### Step 1 - Import Library, Env Variables

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, wrap_model_call, dynamic_prompt, ModelRequest, ModelResponse
from dotenv import load_dotenv
from dataclasses import dataclass
from typing import Literal
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from pprint import pprint
import time
from typing import Any, Callable
from langchain.agents.middleware import before_model, AgentState
from langgraph.runtime import Runtime

# Environment Variables
load_dotenv(override = True)

True

#### Step 2 - Ban List

In [12]:
ban_list = ['politics', 'violence']

#### Step 2 - Middleware

In [17]:
# Logs the model name, latency in ms and token usage for every LLM call.
@wrap_model_call
def trim_logger(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    start = time.perf_counter()
    response = handler(request)
    latency_ms = (time.perf_counter() - start) * 1000
    msg = response.result[-1]
    name = msg.response_metadata.get("model_name", "unknown")
    print("Model name: ", name)
    tokens = (msg.usage_metadata or {}).get("total_tokens", "?")
    print(f"[timing] model={name} latency={latency_ms:.0f}ms tokens={tokens}")
    return response

# If the latest user message contains a word from a banned list, end the run early with a fixed polite refusal and make no LLM call.
@before_model(can_jump_to=["end"])
def topic_guardrail(state: AgentState, runtime: Runtime)->dict[str, Any] | None:
    message = state['messages'][-1].content.lower()
    for i in ban_list:
        if i in message:
            return {
                "messages": [AIMessage("Sorry, I can't help with that. Your message contains a word from ban list")],
                "jump_to": "end",
            }
    return None
        
# Reads context.language ('en' or 'fr') and sets the system prompt so the agent answers in that language.
@dynamic_prompt
def language_prompt(request: ModelRequest) -> str:
    language = request.runtime.context.language
    if language == 'fr':
        return "Respond in french."
    else:
        return "Respond in english."

In [18]:
@dataclass
class Context:
    language: Literal['en', 'fr'] = 'en'

In [19]:
agent = create_agent(model = 'gpt-5-nano', 
                     middleware=[trim_logger, topic_guardrail, language_prompt], 
                     context_schema = Context)

In [20]:
response = agent.invoke({'messages': [HumanMessage('Hi')]}, context = Context(language='fr'))
print(response['messages'][-1].content)

Model name:  gpt-5-nano-2025-08-07
[timing] model=gpt-5-nano-2025-08-07 latency=3547ms tokens=388
Bonjour ! Comment puis-je vous aider aujourd'hui ? Vous pouvez me poser des questions, demander une traduction, une explication, ou de l’aide pour rédiger un texte. Dites-moi ce dont vous avez besoin.
